# 07. 차별화 피처 검증 + F1-Score 측정 (v4)

## 목적
1. **피처 유효성 검증** — 구매 시점 피처와 공동구매 패턴 피처가 실제로 재구매와 상관있는지 확인
2. **F1-Score 비교** — v3 베이스라인 vs v4(새 피처 추가) 성능 차이 측정
3. **최적 임계값 탐색** — 클래스 불균형 보정 + validation 기반 threshold 최적화

| | v3 베이스라인 | v4 (이 노트북) |
|---|---|---|
| 데이터 | k-pick_total_v3.csv | k-pick_total_v4.csv |
| 피처 수 | 26개 | 36개 (+10) |
| 클래스 불균형 보정 | 없음 | `scale_pos_weight` 적용 |
| 임계값 | 0.5 (기본값) | validation 기반 최적값 |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

_candidates = [
    Path(r'c:\Users\gksal\capstone-kpick\K-Pick'),
    Path.cwd(),
    Path.cwd().parent,
]
BASE = next((p for p in _candidates if (p / 'data' / 'prep').exists()), None)
if BASE is None:
    raise FileNotFoundError(f'data/prep 폴더를 찾을 수 없습니다. 현재 경로: {Path.cwd()}')

PREP = BASE / 'data' / 'prep'
print(f'BASE : {BASE}')
print(f'PREP : {PREP}')
print('라이브러리 로드 완료')

---
## 1. 데이터 로드

In [ ]:
print('v4 데이터 로드 중...')
_csv = PREP / 'k-pick_total_v4.csv'
if not _csv.exists():
    raise FileNotFoundError(f'v4 파일 없음: {_csv}\n06_feat_advanced.ipynb를 먼저 실행하세요.')

df = pd.read_csv(_csv)
print(f'전체 행   : {len(df):,}')
print(f'전체 컬럼 : {df.shape[1]}')
print(f'\nlabel 분포:')
print(df['label'].value_counts().rename({0: '미구매(0)', 1: '재구매(1)'}))
print(f'\n재구매율  : {df["label"].mean():.4f} ({df["label"].mean()*100:.2f}%)')

# 새로 추가된 피처 확인
NEW_FEATURES = [
    'up_avg_purchase_interval', 'up_std_purchase_interval',
    'up_purchase_regularity', 'up_interval_count',
    'days_until_expected', 'timing_ratio', 'is_overdue',
    'copurchase_ratio', 'copurchase_weighted_score', 'copurchase_match_count'
]
missing_new = [c for c in NEW_FEATURES if c not in df.columns]
if missing_new:
    print(f'\n누락된 새 피처: {missing_new}')
else:
    print(f'\n새 피처 {len(NEW_FEATURES)}개 모두 존재 확인')

---
## 2. 피처 유효성 검증

### 2-1. 구매 시점 피처 — 재구매(1) vs 미구매(0) 분포 비교

In [ ]:
# label별 새 피처 평균값 비교
SEP = '=' * 55
print(SEP)
print('  새 피처 평균값: 재구매(1) vs 미구매(0)')
print(SEP)
comparison = df.groupby('label')[NEW_FEATURES].mean().T
comparison.columns = ['미구매(0)', '재구매(1)']
comparison['차이(1-0)'] = comparison['재구매(1)'] - comparison['미구매(0)']
comparison['방향'] = comparison['차이(1-0)'].apply(lambda x: '재구매↑' if x > 0 else '재구매↓')
print(comparison.round(4).to_string())

In [ ]:
# 구매 시점 피처 시각화
timing_feats = ['timing_ratio', 'days_until_expected', 'up_purchase_regularity', 'is_overdue']

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
colors = {0: 'steelblue', 1: 'tomato'}
labels_map = {0: '미구매(0)', 1: '재구매(1)'}

# timing_ratio 분포
for lbl in [0, 1]:
    subset = df[df['label'] == lbl]['timing_ratio'].clip(0, 3)
    axes[0].hist(subset, bins=40, alpha=0.6, color=colors[lbl],
                 label=labels_map[lbl], density=True)
axes[0].axvline(1.0, color='black', linestyle='--', linewidth=1, label='주기 도달(1.0)')
axes[0].set_title('timing_ratio 분포')
axes[0].set_xlabel('timing_ratio (경과일 / 평균주기)')
axes[0].legend(fontsize=8)

# days_until_expected 분포
for lbl in [0, 1]:
    subset = df[df['label'] == lbl]['days_until_expected'].clip(-30, 30)
    axes[1].hist(subset, bins=40, alpha=0.6, color=colors[lbl],
                 label=labels_map[lbl], density=True)
axes[1].axvline(0, color='black', linestyle='--', linewidth=1, label='예상 구매일(0)')
axes[1].set_title('days_until_expected 분포')
axes[1].set_xlabel('예상 구매까지 남은 일수 (음수=초과)')
axes[1].legend(fontsize=8)

# up_purchase_regularity 분포
for lbl in [0, 1]:
    subset = df[df['label'] == lbl]['up_purchase_regularity']
    axes[2].hist(subset, bins=30, alpha=0.6, color=colors[lbl],
                 label=labels_map[lbl], density=True)
axes[2].set_title('up_purchase_regularity 분포')
axes[2].set_xlabel('구매 규칙성 (높을수록 규칙적)')
axes[2].legend(fontsize=8)

# is_overdue별 재구매율
overdue_rate = df.groupby('is_overdue')['label'].mean()
bars = axes[3].bar(['아직 안 됨(0)', '주기 초과(1)'],
                   overdue_rate.values, color=['steelblue', 'tomato'])
for bar, val in zip(bars, overdue_rate.values):
    axes[3].text(bar.get_x() + bar.get_width()/2, val + 0.003,
                 f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')
axes[3].set_title('is_overdue별 재구매율')
axes[3].set_ylabel('재구매율')
axes[3].set_ylim(0, overdue_rate.max() * 1.3)

plt.suptitle('구매 시점 피처 검증', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 2-2. 공동구매 패턴 피처 검증

In [ ]:
# copurchase_match_count 구간별 재구매율
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1) copurchase_match_count별 재구매율
cp_rate = df.groupby('copurchase_match_count')['label'].mean()
axes[0].bar(cp_rate.index.astype(str), cp_rate.values, color='mediumpurple')
axes[0].set_title('공동구매 매칭 수별 재구매율')
axes[0].set_xlabel('copurchase_match_count (0~10)')
axes[0].set_ylabel('재구매율')
for x, y in zip(range(len(cp_rate)), cp_rate.values):
    axes[0].text(x, y + 0.002, f'{y:.3f}', ha='center', fontsize=8)

# 2) copurchase_ratio 분포
for lbl in [0, 1]:
    subset = df[df['label'] == lbl]['copurchase_ratio']
    axes[1].hist(subset, bins=25, alpha=0.6,
                 color=colors[lbl], label=labels_map[lbl], density=True)
axes[1].set_title('copurchase_ratio 분포')
axes[1].set_xlabel('공동구매 파트너 매칭 비율 (0~1)')
axes[1].legend(fontsize=8)

# 3) copurchase_weighted_score 분포
for lbl in [0, 1]:
    subset = df[df['label'] == lbl]['copurchase_weighted_score']
    axes[2].hist(subset, bins=25, alpha=0.6,
                 color=colors[lbl], label=labels_map[lbl], density=True)
axes[2].set_title('copurchase_weighted_score 분포')
axes[2].set_xlabel('가중 공동구매 친화도 점수')
axes[2].legend(fontsize=8)

plt.suptitle('공동구매 패턴 피처 검증', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 3. 데이터 분리 (7:2:1) + 클래스 불균형 보정

In [ ]:
DROP_COLS = ['user_id', 'product_id', 'order_id', 'label']
X = df.drop(columns=[c for c in DROP_COLS if c in df.columns])
y = df['label']

# 결측치 처리
missing = X.isnull().sum()
missing = missing[missing > 0]
if len(missing):
    print(f'결측치 발견:\n{missing}')
    X = X.fillna(X.median(numeric_only=True))
    print('→ 중앙값으로 대체 완료')
else:
    print('결측치 없음')

print(f'\n전체 피처 수: {X.shape[1]}')
print(f'  - 기존 피처 (v3) : {X.shape[1] - len(NEW_FEATURES)}개')
print(f'  - 새 피처 (v4)   : {len(NEW_FEATURES)}개')

# 7:2:1 분리
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=1/3, random_state=42, stratify=y_temp
)

print(f'\nTrain : {len(X_train):>8,}행  재구매율: {y_train.mean():.4f}')
print(f'Val   : {len(X_val):>8,}행  재구매율: {y_val.mean():.4f}')
print(f'Test  : {len(X_test):>8,}행  재구매율: {y_test.mean():.4f}')

# 클래스 불균형 비율
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count
print(f'\nscale_pos_weight: {scale_pos_weight:.2f}  (음성:{neg_count:,} / 양성:{pos_count:,})')

---
## 4. LightGBM 학습 (클래스 불균형 보정 적용)

In [ ]:
print('=== LightGBM 학습 중 (v4 피처 + scale_pos_weight) ===')
model_v4 = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=20,
    scale_pos_weight=scale_pos_weight,   # 클래스 불균형 보정
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
model_v4.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=100)
    ]
)

train_acc = accuracy_score(y_train, model_v4.predict(X_train))
val_acc   = accuracy_score(y_val,   model_v4.predict(X_val))
print(f'\nTrain Accuracy : {train_acc:.4f}')
print(f'Val   Accuracy : {val_acc:.4f}')
print(f'최적 트리 수   : {model_v4.best_iteration_}')

---
## 5. 최적 임계값(threshold) 탐색 — validation 세트 기준

In [ ]:
prob_val = model_v4.predict_proba(X_val)[:, 1]

thresholds = np.arange(0.05, 0.55, 0.01)
results_thr = []
for thr in thresholds:
    pred = (prob_val >= thr).astype(int)
    results_thr.append({
        'threshold' : round(thr, 2),
        'f1'        : f1_score(y_val, pred),
        'precision' : precision_score(y_val, pred),
        'recall'    : recall_score(y_val, pred),
    })

thr_df = pd.DataFrame(results_thr)
best_row = thr_df.loc[thr_df['f1'].idxmax()]
BEST_THR = best_row['threshold']

print(f'최적 임계값 : {BEST_THR}')
print(f'Val F1      : {best_row["f1"]:.4f}')
print(f'Precision   : {best_row["precision"]:.4f}')
print(f'Recall      : {best_row["recall"]:.4f}')

# 임계값별 F1 곡선
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(thr_df['threshold'], thr_df['f1'],        label='F1-Score',  color='tomato',      linewidth=2)
ax.plot(thr_df['threshold'], thr_df['precision'], label='Precision', color='steelblue',   linestyle='--')
ax.plot(thr_df['threshold'], thr_df['recall'],    label='Recall',    color='mediumseagreen', linestyle='--')
ax.axvline(BEST_THR, color='black', linestyle=':', linewidth=1.5,
           label=f'최적 threshold = {BEST_THR}')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('임계값별 F1 / Precision / Recall (Validation 세트)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. Test 세트 최종 성능 평가

In [ ]:
prob_test = model_v4.predict_proba(X_test)[:, 1]

# 기본 임계값(0.5) vs 최적 임계값 비교
for thr, label in [(0.5, '기본 (threshold=0.50)'), (BEST_THR, f'최적 (threshold={BEST_THR})')]:
    y_pred = (prob_test >= thr).astype(int)
    sep = '=' * 50
    print(f'\n{sep}')
    print(f'  {label}')
    print(sep)
    print(f'Accuracy  : {accuracy_score(y_test, y_pred):.4f}')
    print(f'Precision : {precision_score(y_test, y_pred):.4f}')
    print(f'Recall    : {recall_score(y_test, y_pred):.4f}')
    print(f'F1-Score  : {f1_score(y_test, y_pred):.4f}')
    print()
    print(classification_report(y_test, y_pred, target_names=['미구매(0)', '재구매(1)']))

In [ ]:
# 혼동 행렬 (최적 임계값 기준)
y_pred_best = (prob_test >= BEST_THR).astype(int)

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test, y_pred_best),
    display_labels=['미구매(0)', '재구매(1)']
)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'혼동 행렬 (threshold={BEST_THR})')
plt.tight_layout()
plt.show()

---
## 7. v3 베이스라인 vs v4 F1-Score 비교

In [ ]:
# v3 베이스라인 재현 (새 피처 제외, threshold=0.5, 불균형 보정 없음)
print('=== v3 베이스라인 학습 중 ===')
OLD_FEATURES = [c for c in X.columns if c not in NEW_FEATURES]

model_v3 = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05, num_leaves=63,
    min_child_samples=20, random_state=42, n_jobs=-1, verbose=-1
)
model_v3.fit(
    X_train[OLD_FEATURES], y_train,
    eval_set=[(X_val[OLD_FEATURES], y_val)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=999)
    ]
)

# v3: 기본 임계값 0.5
y_pred_v3 = model_v3.predict(X_test[OLD_FEATURES])
f1_v3 = f1_score(y_test, y_pred_v3)

# v4: 최적 임계값
f1_v4 = f1_score(y_test, y_pred_best)

# v4: 최적 임계값 없이 기본값 비교도 추가
f1_v4_default = f1_score(y_test, (prob_test >= 0.5).astype(int))

print('\n' + '=' * 55)
print('  F1-Score 비교 결과 (Test 세트)')
print('=' * 55)
print(f'v3 베이스라인 (threshold=0.50) : {f1_v3:.4f}')
print(f'v4 + 불균형보정 (threshold=0.50) : {f1_v4_default:.4f}  (Δ {f1_v4_default - f1_v3:+.4f})')
print(f'v4 + 불균형보정 + 최적 threshold : {f1_v4:.4f}  (Δ {f1_v4 - f1_v3:+.4f})')
print('=' * 55)

In [ ]:
# 비교 차트
fig, ax = plt.subplots(figsize=(8, 4))
models  = ['v3\n(기본)', 'v4+불균형보정\n(thr=0.5)', f'v4+불균형보정\n(thr={BEST_THR})']
scores  = [f1_v3, f1_v4_default, f1_v4]
bar_colors = ['lightgray', 'steelblue', 'tomato']

bars = ax.bar(models, scores, color=bar_colors, edgecolor='black', linewidth=0.8)
for bar, val in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.005,
            f'{val:.4f}', ha='center', fontsize=11, fontweight='bold')

ax.set_ylim(0, max(scores) * 1.2)
ax.set_ylabel('F1-Score')
ax.set_title('모델 버전별 F1-Score 비교 (Test 세트)', fontsize=13)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
## 8. 피처 중요도 — 새 피처가 예측에 기여하는가?

In [ ]:
importance_df = pd.DataFrame({
    'feature'   : X.columns,
    'importance': model_v4.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

importance_df['importance_pct'] = (
    importance_df['importance'] / importance_df['importance'].sum() * 100
).round(2)
importance_df['is_new'] = importance_df['feature'].isin(NEW_FEATURES)

print('=' * 55)
print('  전체 피처 중요도 순위 (★ = 새 피처)')
print('=' * 55)
for _, row in importance_df.iterrows():
    mark = ' ★' if row['is_new'] else '  '
    print(f"{mark} {row['feature']:<35} {row['importance_pct']:>5.2f}%")

# 새 피처 누적 중요도
new_pct = importance_df[importance_df['is_new']]['importance_pct'].sum()
print(f'\n새 피처 10개의 누적 중요도: {new_pct:.2f}%')

In [ ]:
# 피처 중요도 시각화 (새 피처 강조)
fig, ax = plt.subplots(figsize=(11, 8))

bar_colors = [
    'tomato' if row['is_new'] else 'steelblue'
    for _, row in importance_df[::-1].iterrows()
]
ax.barh(
    importance_df['feature'][::-1],
    importance_df['importance'][::-1],
    color=bar_colors
)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='tomato',    label='새 피처 (v4 추가)'),
    Patch(color='steelblue', label='기존 피처 (v3)')
], loc='lower right')

ax.set_title('LightGBM 피처 중요도 (빨강=새 피처)', fontsize=13)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()